In [7]:
import pandas as pd
import geopandas as gpd

pop_master = gpd.read_file('population_master.shp')

print("=== pop_master ===")
print("행 수:", len(pop_master))
print("grid_id 중복 수:", pop_master['grid_id'].duplicated().sum())
print("grid_id dtype:", pop_master['grid_id'].dtype)
print(pop_master.columns.tolist())

=== pop_master ===
행 수: 11388
grid_id 중복 수: 0
grid_id dtype: int64
['grid_id', 'm_10g_pop', 'm_20g_pop', 'm_30g_pop', 'm_40g_pop', 'm_50g_pop', 'm_60g_pop', 'w_10g_pop', 'w_20g_pop', 'w_30g_pop', 'w_40g_pop', 'w_50g_pop', 'w_60g_pop', 'work_00h', 'work_01h', 'work_02h', 'work_03h', 'work_04h', 'work_05h', 'work_06h', 'work_07h', 'work_08h', 'work_09h', 'work_10h', 'work_11h', 'work_12h', 'work_13h', 'work_14h', 'work_15h', 'work_16h', 'work_17h', 'work_18h', 'work_19h', 'work_20h', 'work_21h', 'work_22h', 'work_23h', 'visi_00h', 'visi_01h', 'visi_02h', 'visi_03h', 'visi_04h', 'visi_05h', 'visi_06h', 'visi_07h', 'visi_08h', 'visi_09h', 'visi_10h', 'visi_11h', 'visi_12h', 'visi_13h', 'visi_14h', 'visi_15h', 'visi_16h', 'visi_17h', 'visi_18h', 'visi_19h', 'visi_20h', 'visi_21h', 'visi_22h', 'visi_23h', 'wd_w_pop', 'wd_v_pop', 'we_w_pop', 'we_v_pop', 'pop_peak_h', 'pop_elderl', 'pop_lunch_', 'geometry']


In [8]:
facility_df = pd.read_csv('최종최종_분당구30m격자_분석데이터.csv')

print("facility_df 행 수:", len(facility_df))
print("facility_df grid_id 중복 수:", facility_df['grid_id'].duplicated().sum())
print("facility_df grid_id 유니크 수:", facility_df['grid_id'].nunique())

facility_df 행 수: 32124
facility_df grid_id 중복 수: 0
facility_df grid_id 유니크 수: 32124


In [9]:
pop_ids = set(pop_master['grid_id'])
facility_ids = set(facility_df['grid_id'])

print("pop_master 격자 수:", len(pop_ids))
print("facility_df 격자 수:", len(facility_ids))
print("교집합:", len(pop_ids & facility_ids))
print("pop_master에만 있는 격자:", len(pop_ids - facility_ids))
print("facility_df에만 있는 격자:", len(facility_ids - pop_ids))

pop_master 격자 수: 11388
facility_df 격자 수: 32124
교집합: 11388
pop_master에만 있는 격자: 0
facility_df에만 있는 격자: 20736


In [11]:
combined = pd.merge(
    pop_master,
    facility_df.drop(columns=['WKT', 'WKT.1']),
    on='grid_id',
    how='left'
)

print("merge 후 행 수:", len(combined))
print("merge 후 컬럼 수:", len(combined.columns))
print(combined.columns.tolist())

# 시설 컬럼 결측치 확인 (있으면 안 되는 게 정상)
facility_cols = ['park_cnt','bus_cnt','subway_cnt','shade_cnt','senior_center_cnt',
                  'elder_class_cnt','elder_welfare_cnt','cooling_shelter_cnt',
                  'daycare_cnt','teen_fac_cnt','smart_shelter_cnt','tree_road_len','tree_road_cnt']
print("\n시설 컬럼별 결측치 수:")
print(combined[facility_cols].isna().sum())

merge 후 행 수: 11388
merge 후 컬럼 수: 82
['grid_id', 'm_10g_pop', 'm_20g_pop', 'm_30g_pop', 'm_40g_pop', 'm_50g_pop', 'm_60g_pop', 'w_10g_pop', 'w_20g_pop', 'w_30g_pop', 'w_40g_pop', 'w_50g_pop', 'w_60g_pop', 'work_00h', 'work_01h', 'work_02h', 'work_03h', 'work_04h', 'work_05h', 'work_06h', 'work_07h', 'work_08h', 'work_09h', 'work_10h', 'work_11h', 'work_12h', 'work_13h', 'work_14h', 'work_15h', 'work_16h', 'work_17h', 'work_18h', 'work_19h', 'work_20h', 'work_21h', 'work_22h', 'work_23h', 'visi_00h', 'visi_01h', 'visi_02h', 'visi_03h', 'visi_04h', 'visi_05h', 'visi_06h', 'visi_07h', 'visi_08h', 'visi_09h', 'visi_10h', 'visi_11h', 'visi_12h', 'visi_13h', 'visi_14h', 'visi_15h', 'visi_16h', 'visi_17h', 'visi_18h', 'visi_19h', 'visi_20h', 'visi_21h', 'visi_22h', 'visi_23h', 'wd_w_pop', 'wd_v_pop', 'we_w_pop', 'we_v_pop', 'pop_peak_h', 'pop_elderl', 'pop_lunch_', 'geometry', 'park_cnt', 'bus_cnt', 'subway_cnt', 'shade_cnt', 'senior_center_cnt', 'elder_class_cnt', 'elder_welfare_cnt', 'coolin

In [12]:
import geopandas as gpd

if not isinstance(combined, gpd.GeoDataFrame):
    combined = gpd.GeoDataFrame(combined, geometry='geometry', crs=pop_master.crs)

combined.to_file('grid_master_final.shp', driver='ESRI Shapefile', encoding='utf-8')

print("저장 완료: grid_master_final.shp")
print(f"행 수: {len(combined):,}개, 컬럼 수: {len(combined.columns)}개")

saved_check = gpd.read_file('grid_master_final.shp')
print("\n저장 후 실제 컬럼명:")
print(saved_check.columns.tolist())

/tmp/ipykernel_27/1410383290.py:6: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  combined.to_file('grid_master_final.shp', driver='ESRI Shapefile', encoding='utf-8')


저장 완료: grid_master_final.shp
행 수: 11,388개, 컬럼 수: 82개

저장 후 실제 컬럼명:
['grid_id', 'm_10g_pop', 'm_20g_pop', 'm_30g_pop', 'm_40g_pop', 'm_50g_pop', 'm_60g_pop', 'w_10g_pop', 'w_20g_pop', 'w_30g_pop', 'w_40g_pop', 'w_50g_pop', 'w_60g_pop', 'work_00h', 'work_01h', 'work_02h', 'work_03h', 'work_04h', 'work_05h', 'work_06h', 'work_07h', 'work_08h', 'work_09h', 'work_10h', 'work_11h', 'work_12h', 'work_13h', 'work_14h', 'work_15h', 'work_16h', 'work_17h', 'work_18h', 'work_19h', 'work_20h', 'work_21h', 'work_22h', 'work_23h', 'visi_00h', 'visi_01h', 'visi_02h', 'visi_03h', 'visi_04h', 'visi_05h', 'visi_06h', 'visi_07h', 'visi_08h', 'visi_09h', 'visi_10h', 'visi_11h', 'visi_12h', 'visi_13h', 'visi_14h', 'visi_15h', 'visi_16h', 'visi_17h', 'visi_18h', 'visi_19h', 'visi_20h', 'visi_21h', 'visi_22h', 'visi_23h', 'wd_w_pop', 'wd_v_pop', 'we_w_pop', 'we_v_pop', 'pop_peak_h', 'pop_elderl', 'pop_lunch_', 'park_cnt', 'bus_cnt', 'subway_cnt', 'shade_cnt', 'senior_cen', 'elder_clas', 'elder_welf', 'coolin

In [14]:
import geopandas as gpd
import pandas as pd

# 1. 팀원 격자 기준 파일 (04번 원본 or grid30m_merged.csv)
grid_env = pd.read_csv('grid30m_merged.csv', usecols=['grid_id', 'lon', 'lat'])

# 2. 팀원 인구·시설 통합 데이터 (population_master 또는 merge 후 파일)
pop_master = gpd.read_file('population_master.shp')  # or merge 후 파일명

# --- grid_id 값 범위/타입 비교 ---
print("팀원 grid_id 범위:", grid_env['grid_id'].min(), "~", grid_env['grid_id'].max())
print("팀원 grid_id 범위:", pop_master['grid_id'].min(), "~", pop_master['grid_id'].max())
print("팀원 grid_id dtype:", grid_env['grid_id'].dtype)
print("팀원 grid_id dtype:", pop_master['grid_id'].dtype)

# --- 직접 교집합 확인 (숫자값 그대로 같은 체계인지) ---
common = set(grid_env['grid_id']) & set(pop_master['grid_id'])
print("직접 grid_id 교집합 개수:", len(common))

# --- 혹시 값 체계가 다르면, 좌표(중심점) 기준으로 같은 격자인지 재확인 ---
pop_master_pt = pop_master.copy()
pop_master_pt['centroid'] = pop_master_pt.geometry.centroid
pop_master_pt['lon'] = pop_master_pt['centroid'].x
pop_master_pt['lat'] = pop_master_pt['centroid'].y

# 좌표 반올림 후 매칭 (부동소수점 오차 방지)
grid_env['lon_r'] = grid_env['lon'].round(5)
grid_env['lat_r'] = grid_env['lat'].round(5)
pop_master_pt['lon_r'] = pop_master_pt['lon'].round(5)
pop_master_pt['lat_r'] = pop_master_pt['lat'].round(5)

coord_merge = pd.merge(
    grid_env[['grid_id','lon_r','lat_r']],
    pop_master_pt[['grid_id','lon_r','lat_r']],
    on=['lon_r','lat_r'], suffixes=('_env','_pop')
)
print("좌표 기준 매칭 격자 수:", len(coord_merge))
print(coord_merge.head())

팀원 grid_id 범위: 1 ~ 54029
팀원 grid_id 범위: 21907 ~ 54024
팀원 grid_id dtype: int64
팀원 grid_id dtype: int64
직접 grid_id 교집합 개수: 11388
좌표 기준 매칭 격자 수: 11270
   grid_id_env      lon_r     lat_r  grid_id_pop
0           21907  127.12401  37.41705         21907
1           21909  127.12469  37.41706         21909
2           21910  127.12503  37.41706         21910
3           21912  127.12570  37.41707         21912
4           21914  127.12638  37.41708         21914


/tmp/ipykernel_27/2666351568.py:22: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pop_master_pt['centroid'] = pop_master_pt.geometry.centroid


In [15]:
# 1) pop_master 불러오기
pop_master = gpd.read_file('population_master.shp')

# 2) facility_df 불러오기 (이 부분 코드가 안 보였는데, 07번 노트북 앞쪽에 있을 거야)
facility_df = pd.read_csv('최종최종_분당구30m격자_분석데이터.csv')  # 또는 원래 쓰던 방식대로

# 3) combined 만들기 (지난번에 돌렸던 그 코드)
combined = pd.merge(
    pop_master,
    facility_df.drop(columns=['WKT', 'WKT.1']),
    on='grid_id',
    how='left'
)

# 4) 여기까지 combined가 잘 만들어졌는지 확인
print(len(combined), len(combined.columns))

11388 82


In [16]:
import pandas as pd
import geopandas as gpd

# 1. 팀원 데이터 불러오기 (LST/NDVI/토지피복 등 전부 포함된 파일)
grid_env = pd.read_csv('grid30m_merged.csv')

# 2. combined는 이미 만들어둔 상태 (팀원+팀원, 11,388개, geometry 포함 GeoDataFrame)
#    geometry 컬럼은 팀원 쪽 lon/lat과 겹치니, 최종본엔 하나만 남기기 위해
#    combined의 geometry는 살리고 팀원의 lon/lat은 참고용으로만 남김

# grid_id 기준 left merge (combined=11,388개 기준 → 옵션 A 그대로 유지)
final_merged = pd.merge(
    combined,
    grid_env,
    on='grid_id',
    how='left',
    suffixes=('', '_env')   # 혹시 겹치는 컬럼명 있으면 팀원 쪽에 _env 붙음
)

print("최종 통합 행 수:", len(final_merged))
print("최종 통합 컬럼 수:", len(final_merged.columns))

# 팀원 쪽 데이터가 매칭 안 된 격자 있는지 확인 (있으면 안 되는 게 정상 — combined가 팀원 범위 안에 있으니)
print("\n팀원 데이터(LST) 결측 격자 수:", final_merged['LST_C_mean_5scene'].isna().sum())

# geometry가 두 개(문자열 WKT + geopandas geometry) 섞여 있을 수 있어 정리
if 'geometry' in final_merged.columns:
    final_merged = final_merged.drop(columns=[c for c in ['geometry_env'] if c in final_merged.columns])

# CSV로 저장
final_merged.to_csv('grid_HSI_final_merged.csv', index=False, encoding='utf-8-sig')

print(f"\n저장 완료: grid_HSI_final_merged.csv")
print(f"행 수: {len(final_merged):,}개, 컬럼 수: {len(final_merged.columns)}개")

최종 통합 행 수: 11388
최종 통합 컬럼 수: 212

팀원 데이터(LST) 결측 격자 수: 433

저장 완료: grid_HSI_final_merged.csv
행 수: 11,388개, 컬럼 수: 212개
